# Fine-tune LFM2 Extract on fact claims

Teaches `LiquidAI/LFM2-350M-Extract` to list the checkable facts in one lesson passage (a number, a date or a named thing about one specific, named thing), in the shape the fact checker takes, with a LoRA adapter. Then it merges the adapter and converts the result to ONNX for transformers.js, the same way as the import model's notebook.

The data loads from the published dataset, [playforgecoding/spelling-creator-fact-claims](https://huggingface.co/datasets/playforgecoding/spelling-creator-fact-claims), so there is nothing to upload. To train on data you have just rebuilt instead:

```bash
cd packages/core
node scripts/fact-eval/label.mjs          # only for passages with no label yet (Claude, through the Claude Code CLI)
node scripts/fact-eval/make-dataset.mjs
```

and upload `train.jsonl` and `holdout.jsonl` from `scripts/fact-eval/out/dataset/` with the file panel on the left; files there win over the published dataset.

Runtime: a free T4 is plenty. A passage and its claims are short and the model is small, so training takes a few minutes. Fill in your Hugging Face name in the first cell and run top to bottom.

In [ ]:
import os

BASE_MODEL = "LiquidAI/LFM2-350M-Extract"      # or "LiquidAI/LFM2-1.2B-Extract"
HF_USER = "YOUR_NAME"                         # your Hugging Face username
NAME = BASE_MODEL.split("/")[-1]              # LFM2-350M-Extract or LFM2-1.2B-Extract
MERGED_REPO = f"{HF_USER}/{NAME}-facts"       # merged weights
ONNX_REPO = f"{HF_USER}/{NAME}-facts-ONNX"    # what run.mjs --models takes

# The published dataset, used unless the two files below are uploaded.
DATASET_REPO = "playforgecoding/spelling-creator-fact-claims"
TRAIN_FILE = "train.jsonl"
HOLDOUT_FILE = "holdout.jsonl"
OUTPUT_DIR = f"{NAME}-facts"

# The data is small (about 350 passages), so more epochs than the import
# model's three; the held-out loss after each epoch shows when to stop.
EPOCHS = 5
LEARNING_RATE = 2e-4
LORA_RANK = 16
MAX_LENGTH = 3072   # the rules and schema are about 1.7k tokens, a passage and its claims a few hundred

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

In [ ]:
# Colab ships an old torchao that transformers 5 refuses to import beside; this
# notebook never uses it, so it goes.
#
# Torch stays at the version Colab ships. If a dependency upgrades it while a
# torch from before is loaded, the session breaks with "module 'torch' has no
# attribute '_utils'"; the version is read without importing torch, so this
# cell is safe to rerun. If that error shows anyway: Runtime > Restart session,
# then run on from the first cell, skipping this one.
import importlib.metadata

with open("torch-pin.txt", "w") as f:
    f.write(f"torch=={importlib.metadata.version('torch')}\n")
%pip uninstall -q -y torchao
%pip install -q -c torch-pin.txt "transformers>=4.56" "trl>=0.21" peft datasets accelerate huggingface_hub

## Load the model and the data

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "Pick a GPU runtime: Runtime > Change runtime type"
# Real bfloat16 needs an Ampere GPU or newer (A100, L4). is_bf16_supported()
# also says yes on a T4, which only emulates it, and TRL's loss kernel then
# fails with "gemm input type at::BFloat16 ... compute capability 8.0".
BF16 = torch.cuda.get_device_capability(0)[0] >= 8
DTYPE = torch.bfloat16 if BF16 else torch.float16
print(torch.cuda.get_device_name(0), "bfloat16" if BF16 else "float16")

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
try:
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=DTYPE, device_map="auto")
except TypeError:  # transformers 4.x spells it torch_dtype
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=DTYPE, device_map="auto")

# Uploaded files win; otherwise the published dataset, whose "test" split is
# the same held-out lessons as holdout.jsonl.
if os.path.exists(TRAIN_FILE) and os.path.exists(HOLDOUT_FILE):
    raw = load_dataset("json", data_files={"train": TRAIN_FILE, "test": HOLDOUT_FILE})
    print("using the uploaded files")
else:
    raw = load_dataset(DATASET_REPO)
    print("using", DATASET_REPO)
raw = raw.select_columns(["messages"])

# As prompt and completion, so only the reply carries loss: the rules and
# schema in the system turn are the same in every example, and are most of
# its tokens. Works with any chat template, unlike assistant_only_loss.
data = raw.map(
    lambda r: {"prompt": r["messages"][:2], "completion": r["messages"][2:]},
    remove_columns=["messages"],
)
print(data)

In [ ]:
import torch.nn as nn

# The linear layers to put adapters on. LFM2 mixes attention blocks with gated
# convolution blocks, so this finds the names rather than guessing.
names = sorted({n.split(".")[-1] for n, m in model.named_modules() if isinstance(m, nn.Linear)})
TARGET_MODULES = [n for n in names if n != "lm_head"]
print(TARGET_MODULES)

## Train

A checkpoint is written every epoch. If Colab disconnects, rerun the cells above and this one with `RESUME = True`.

In [ ]:
import math
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

RESUME = False
GRAD_ACCUM = 4

# Warm up over the first 5% of steps, given as a step count: transformers 5.x
# dropped warmup_ratio, and an integer works on 4.x and 5.x alike.
WARMUP_STEPS = max(1, round(0.05 * EPOCHS * math.ceil(len(data["train"]) / GRAD_ACCUM)))

lora = LoraConfig(
    r=LORA_RANK,
    lora_alpha=2 * LORA_RANK,
    lora_dropout=0.05,
    target_modules=TARGET_MODULES,
    task_type="CAUSAL_LM",
)

cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    max_length=MAX_LENGTH,
    gradient_checkpointing=True,
    bf16=BF16,
    fp16=not BF16,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=cfg,
    train_dataset=data["train"],
    eval_dataset=data["test"],
    processing_class=tok,
    peft_config=lora,
)

resume = RESUME and os.path.isdir(OUTPUT_DIR) and any(d.startswith("checkpoint-") for d in os.listdir(OUTPUT_DIR))
trainer.train(resume_from_checkpoint=resume or None)

## Score the held-out passages

A rough version of what `score.mjs` does, enough to decide whether to export. A claim counts as found when a labelled one has the same property and an overlapping quote, or the same value. A quote that is not in the passage is invented (the app drops those, but there should be none). "Quiet when none" is how often the model lists nothing for a passage with no checkable fact. All held-out passages, about 24, a few seconds each.

In [ ]:
import json

def generate(messages, max_new_tokens=1000):
    # Render the template to text first: transformers 5.x returns a dict from
    # apply_chat_template where 4.x returned a tensor, and this works on both.
    text = tok.apply_chat_template(messages[:2], add_generation_prompt=True, tokenize=False)
    prompt = tok(text, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][prompt.shape[1]:], skip_special_tokens=True)

def parse(reply):
    # None for anything but a list of claims, a shape of the model's own included.
    reply = reply.strip().removeprefix("```json").removeprefix("```").removesuffix("```")
    try:
        claims = json.loads(reply[reply.find("{"):]).get("claims")
    except Exception:
        return None
    return claims if isinstance(claims, list) else None

squash = lambda s: " ".join(str(s or "").split()).lower()

def same(g, w):
    if g.get("property") != w["property"]:
        return False
    a, b = squash(g.get("quote")), squash(w["quote"])
    if a and b and (a in b or b in a):
        return True
    if w["stated"]:
        return squash(g.get("stated")) == squash(w["stated"])
    try:
        return abs(float(g.get("value")) - w["value"]) <= abs(w["value"]) * 0.005
    except (TypeError, ValueError):
        return False

model.eval()
t = dict(passages=0, parsed=0, want=0, got=0, found=0, invented=0, quiet=0, quiet_right=0)
for i, row in enumerate(raw["test"], 1):
    passage = squash(row["messages"][1]["content"])
    want = json.loads(row["messages"][2]["content"])["claims"]
    got = parse(generate(row["messages"]))
    t["passages"] += 1
    t["want"] += len(want)
    if not want:
        t["quiet"] += 1
        t["quiet_right"] += got == []
    if got is None:
        continue
    t["parsed"] += 1
    real = [g for g in got if isinstance(g, dict) and squash(g.get("quote")) and squash(g.get("quote")) in passage]
    t["invented"] += len(got) - len(real)
    t["got"] += len(real)
    for w in want:
        k = next((k for k, g in enumerate(real) if same(g, w)), None)
        if k is not None:
            real.pop(k)
            t["found"] += 1
    print(f"{i}/{len(raw['test'])} passages", end="\r")

pct = lambda a, b: f"{100 * a / b:.0f}%" if b else "n/a"
print()
print("replies parsed   ", pct(t["parsed"], t["passages"]))
print("recall           ", pct(t["found"], t["want"]), f"({t['found']} of {t['want']} labelled claims)")
print("precision        ", pct(t["found"], t["got"]), f"({t['got']} listed)")
print("invented quotes  ", t["invented"], "(should be 0)")
print("quiet when none  ", pct(t["quiet_right"], t["quiet"]))

## Merge the adapter and push the merged model

In [ ]:
from huggingface_hub import login

login()

# Public, like the import model: trained on published lessons, and a public
# repo is what run.mjs, the app and the conversion step can fetch without a
# token.
merged = trainer.model.merge_and_unload()
merged.save_pretrained("merged", safe_serialization=True)
tok.save_pretrained("merged")
merged.push_to_hub(MERGED_REPO, private=False)
tok.push_to_hub(MERGED_REPO, private=False)
print("pushed", MERGED_REPO)

## Convert to ONNX for transformers.js

The same route as the import model (see that notebook and the docs page): Microsoft's onnxruntime-genai model builder, then `relayout-onnx.py` to fix the cache names, a symbolic cache dimension and the chat template's location. `q4f16` for WebGPU and `int8` for a CPU.

In [ ]:
%pip install -q onnxruntime-genai onnx onnx_ir onnxscript

import subprocess, sys

BUILDS = (("webgpu", "int4", "q4f16"), ("cpu", "int8", "int8"))
for provider, precision, _ in BUILDS:
    print("building", precision, "for", provider)
    subprocess.run(
        [sys.executable, "-m", "onnxruntime_genai.models.builder",
         "-i", "merged", "-o", f"/content/build-{provider}", "-p", precision, "-e", provider, "-c", "/content/hf-cache"],
        check=True,
    )

In [ ]:
import urllib.request

# The relayout script from the repo (packages/core/scripts/extract-eval/relayout-onnx.py).
# Upload it beside the notebook with the file panel; failing that, this fetches it
# from main, which works once the branch carrying it has been merged.
RELAYOUT = "relayout-onnx.py"
if not os.path.exists(RELAYOUT):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Spelling-Creator/spelling-creator/main/packages/core/scripts/extract-eval/relayout-onnx.py",
        RELAYOUT,
    )

DEST = "/content/onnx-repo"
for provider, _, dtype in BUILDS:
    subprocess.run([sys.executable, RELAYOUT, f"/content/build-{provider}", DEST, dtype, "merged"], check=True)
print(os.listdir(DEST), os.listdir(f"{DEST}/onnx"))

from huggingface_hub import HfApi

api = HfApi()
api.create_repo(ONNX_REPO, private=False, exist_ok=True)
api.upload_folder(folder_path=DEST, repo_id=ONNX_REPO)
print("pushed", ONNX_REPO)

## Measure it with the real scorer

Back on your machine, against the held-out lessons:

```bash
cd packages/core
node scripts/fact-eval/run.mjs --holdout --dtype int8 --models YOUR_NAME/LFM2-350M-Extract-facts-ONNX > ../../logs/fact-finetuned-int8.log 2>&1
```

and compare with the stock models' rows in the docs page (Monorepo, Fact claim model).